### 视觉与语言（Multi-Model Foundation Models）
前面介绍的各种模型都是为某一特定任务专门设计的，而这里要将的 Foundation Model 则有以下特征：泛用于多个不同任务；通常参数规模和数据集规模巨大；通常经过某种任务的无监督学习。简单来说，这类模型把图像、文本、音频、视频等不同模态的数据，映射到一个共享的语义空间里，让模型能跨模态理解和匹配
#### CLIP
* CLIP：CLIP 先用大量图文对进行对比学习，训练一个图像编码器和一个文本编码器，使匹配的图文在共享特征空间中相近，不匹配的图文相远。下游做分类时，把未知图片编码成图像向量，把候选类别写成文本 prompt 后编码成文本向量，然后比较它们的相似度，选择最相似的文本类别作为图片预测结果（分类时，用若干短语标注类别往往比单个词更有效），这就实现了零样本分类<br>
CLIP成功的原因在于通过应用 Transformer ，将参数规模扩大，以及规模巨大的训练数据集
* CoCa：它也是先给出大量图文对作为训练集，但相比 CLIP 只做图文对比学习，CoCa 同时加入了 captioning 任务。具体来说，它一方面通过 image encoder 得到图片特征，通过 text decoder 得到文字特征，并用 contrastive loss 让匹配的图文向量靠近、不匹配的图文向量远离；另一方面，它又加入一个 multimodal text decoder，通过 cross-attention 读取图片特征，学习根据图片逐词生成对应的文本描述。这样训练后，CoCa 不仅像 CLIP 一样可以在下游分类、检索任务中比较图片向量和文本向量的相似度，还可以完成图像描述生成等需要“看图说话”的任务。
<div align="center">
  <img src="class_images/CoCa.jpg" width="500">
</div>

* CLIP类方法的局限：严重依赖批次大小；图片层面的注释对于监督来说是不够的；如何筛选更高效的数据集

#### LM + Vision
* LLaVa：LLaVA 先用预训练好的 CLIP 图像编码器把输入图片编码成图像特征，再通过一个线性层把这些图像特征映射到 LLM 的输入 embedding 空间，形成一串视觉 token；随后将这些视觉 token 和用户输入的文本 token 一起送入 LLaMA 这样的语言模型中，由 LLM 自回归地生成回答。训练时，LLaVA 首先固定 CLIP 和 LLM，只训练中间的线性投影层，让图像特征能够对接语言模型；之后再微调 LLM 和投影层，使模型能够完成看图问答、图像描述和多模态对话等任务
<div align="center">
  <img src="class_images/LLaVA.jpg" width="500">
</div>

* Flamingo：Flamingo 使用预训练视觉编码器提取图像特征，并在预训练语言模型的若干层中插入 gated cross-attention 模块，使语言 token 能够以 cross-attention 的方式读取图像特征；门控参数控制视觉信息注入语言模型的程度，并且通常初始化为 0，以避免一开始破坏原有语言模型能力。最终模型可以根据图片和文字提示生成回答
<div align="center">
  <img src="class_images/flamingo_gated_crossattention.jpg" width="500">
</div>

* Molmo：相比 LLaVA，Molmo 的特点不主要在于提出了全新的模型结构，而在于它更强调开放性和高质量数据。它使用 PixMo 这类人工构建的数据集，其中包含非常详细的图像描述、自由形式的图像问答以及 pointing 数据，因此模型不仅能描述图片内容，还能更好地理解物体之间的关系，并把语言回答和图像中的具体位置对应起来

#### 更多 Foundation Models
* Segment Anything Model(SAM)：它的目标不是判断图片中有什么类别，而是根据用户给出的提示，例如一个点、一个框或已有的 mask，把图像中对应的物体或区域以像素级别分割出来。SAM 的结构主要包括图像编码器、提示编码器和 mask 解码器：图像编码器提取整张图片的视觉特征，提示编码器理解用户想分割哪里，最后 mask 解码器输出目标区域的分割结果。相比普通检测模型只给出矩形框，SAM 输出的是更精细的物体轮廓，因此可以用于抠图、图像编辑、医学图像、机器人视觉等任务。它的核心特点是 promptable segmentation，也就是“通过提示来分割任意物体”